# LIBRARIES

In [ ]:
import requests
import urllib
import pandas as pd
from datetime import datetime

import numpy as np
import matplotlib.pyplot as plt
import folium
import osmnx as ox
import seaborn as sns
import plotly.express as px

from lib.graph import df_animation_multiple_path
from lib.frontend_functions import (
    login,
    data_api,
    data_microservice
)
from lib.models import OptimizationParams



from lib.data import hexa_color_generator_list
from lib.data import load_excel_data

# %load_ext autoreload
# %autoreload 2


%matplotlib inline
plt.rcParams['figure.figsize'] = (50.0, 25.0)


In [ ]:
import time

def start_timer():
    print("Timer started. Press Enter to stop.")
    start = time.time()
    input()
    end = time.time()
    duration = round(end - start, 2)
    print(f"Elapsed time: {duration} seconds")

# start_timer()

# ENV VARIABLES

In [ ]:
route_id = 2
day = 1
radio = 1500

# Optimization microservice (replaces legacy /api/v1/optimization/* of the monolith).
# Local dev port matches services/optimization/.env (PORT=3120).
url_token = 'https://api.bearsoft.com.bo'
url = 'http://localhost:3120'
# url = 'http://localhost:3100'                  # API Gateway local
# url = 'https://<api-gateway-id>.execute-api.us-east-1.amazonaws.com'  # AWS deploy


# BACKEND

In [ ]:
print("Timer started.")
start_process = time.time()
start_time = datetime.now()
start_format = start_time.strftime('%Y-%m-%d %H:%M:%S')
start_format

## Load Database with data from excel file if it is necessary

In [ ]:
model = {'index': 'id',
    'rutaId': 'route_id',
    'diaId': 'day',
    'clienteId': 'client_id',
    'cliente': 'client',
    'latitud': 'latitude',
    'longitud': 'longitude'
}
table = 'routes'
file_name = 'RutasRios.xlsx'

data_database = load_excel_data(model, table, file_name)

data_database

## Login into SmartDecisions API

In [ ]:
token = login(url_token)
token

## Database

In [ ]:
data = data_api(token, url, 'data_model', OptimizationParams(route_id, day, primary=1))
data


In [ ]:
distances = data_api(token, url, 'distances', OptimizationParams(route_id, day, primary=1))

distances


## Using the optimal distance algorithm between neighbors

In [ ]:
# dtf_final = data_api(token, url, 'optimization', OptimizationParams(route_id, day, primary=1, dist=radio))
dtf_final = data_api(token, url, 'optimal_route', OptimizationParams(route_id, day, primary=1, dist=radio))

dtf_final


## Distances matrix using graphs

In [ ]:
distances = data_api(token, url, 'distance_matrix', OptimizationParams(route_id, day, primary=2))

df_distances = pd.DataFrame(distances)
df_distances


## Final route, with nodes and graphs

In [ ]:
dtf_route = data_api(token, url, 'route', OptimizationParams(route_id, day, primary=1, dist=radio))
dtf_route.index.name = 'visit_order'

dtf_route


In [ ]:
dtf_route.isnull().any().any()

In [ ]:
TAG_NUMBER_STOPS = 'Num stops'
TAG_TOTAL_DISTANCE = 'Distance'
TAG_TOTAL_TIME = 'Duration Time'

n_stops = dtf_route['origin'].size
time_distance = float(dtf_route['time_seg'].sum()/60)
distance = float(dtf_route['distance'].sum())

display(f'{TAG_NUMBER_STOPS}: {n_stops}')
display(f'{TAG_TOTAL_TIME}: {time_distance:.2f} minutes')
display(f'{TAG_TOTAL_DISTANCE}: {distance:.2f} meters')

In [ ]:
end_process = time.time()
duration = round(end_process - start_process, 2)
end_time = datetime.now()
elapsed_time = end_time - start_time
end_format = end_time.strftime('%Y-%m-%d %H:%M:%S')

end_format

In [ ]:
print(f"Elapsed time: {duration} seconds")
elapsed_time

# FRONTEND

## Map by id

In [ ]:
start = data[data.index == 0][['y', 'x']].values[0]
print('starting point:' , start)

In [ ]:
map = folium.Map(location = start, tiles = 'cartodbpositron', zoom_start = 16)
data.apply(lambda row:
    folium.CircleMarker(
        location = [row['y'], row['x']], 
        color = row['color'], fill = True, radius = 8).add_to(map), axis = 1)

map 

## Creating the graph and nodes

In [ ]:
G = ox.graph_from_point(start, dist = radio, network_type = 'drive')
G = ox.add_edge_speeds(G)
G = ox.add_edge_travel_times(G)
fig, ax = ox.plot_graph(G, bgcolor = 'white', node_size = 5 , node_color = 'black', figsize = (16,8))

## Map of the final route from the use of the optimization algorithm

In [ ]:
# Crear un mapa centrado en un punto inicial
route_map = folium.Map(location = start, zoom_start = 16)
route = []
size, _ = dtf_final.shape
# Agregar marcadores para cada cliente
for i, row in dtf_final.iterrows() :
    num = f'<b>NUM:</b> {int(i + 1)}<br>'
    # node = f'<b>NODE:</b> {int(dtf_final.iloc[i]['node'])}<br>'
    orig = f'<b>START:</b> {int(dtf_final.iloc[i]['origin'])}<br>'
    dest = f'<b>END:</b> {int(dtf_final.iloc[i]['target'])}<br>'
    dist = f'<b>DIST:</b> {round(dtf_final.iloc[i]['distance'], 2)} m<br>'
    # popup = f'{num}{node}{orig}{dest}{dist}'
    popup = f'{num}{orig}{dest}{dist}'
    folium.Marker(location = [row['y'], row['x']],
                icon = folium.Icon(color = 'red' if i == 0 else 'green' if i == size - 1 else 'cadetblue', 
                                   icon = 'home' if i == 0 else 'flag' if i == size - 1 else 'info-sign'),
                tooltip = popup).add_to(route_map)
    lines = (row['y'], row['x'])
    route.append(lines)

line = folium.PolyLine([route], color = 'cyan', weight = 3.5, opacity = 0.8)
route_map.add_child(line)

# Mostrar el mapa
route_map

In [ ]:
map_route = folium.Map(location = start, zoom_start = 16)
for stop in dtf_route.itertuples():
    initial_stop = stop.Index == 0
    # marker for current stop
    icon = folium.Icon(icon = 'home' if initial_stop else 'cloud', 
                       color = 'green' if initial_stop else 'lightgray',
                       prefix = 'fa')
    marker = folium.Marker(location = (stop.y, stop.x),
        icon = icon,
        tooltip = f'<b>Name</b>: {stop.origin} <br>' \
              + f'<b>Stop number</b>: {stop.Index} <br>'
    )

    # line for the route segment connecting current to next stop
    line = folium.PolyLine(
        locations = [(stop.y, stop.x), 
                   (stop.y_next, stop.x_next)],
        # display the start, end, and distance of each segment
        tooltip = f'<b>From</b>: {stop.origin} <br>' \
              + f'<b>To</b>: {stop.target} <br>' \
              + f'<b>Time</b>: {stop.time_seg:.0f} m',
    )
    # add elements to the map
    marker.add_to(map_route)
    line.add_to(map_route)

# add route's last marker, as it wasn't included in for loop
folium.Marker(
    location = (stop.y_next, stop.x_next),
    tooltip = f'<b>Name</b>: {stop.target} <br>' \
          + f'<b>Stop number</b>: {stop.Index + 1} <br>', 
    icon = folium.Icon(icon = 'info-sign', color = 'cadetblue')
).add_to(map_route)

map_route  # show map

## Map from start to end

In [ ]:
map_start_end = folium.Map(location = start, zoom_start = 16)
route = []
size, _ = dtf_final.shape

for i, row in dtf_final.iterrows() :
    num = f'<b>NUM:</b> {int(i + 1)}<br>'
    # node = f'<b>NODE:</b> {int(dtf_final.iloc[i]['node'])}<br>'
    orig = f'<b>START:</b> {int(dtf_final.iloc[i]['origin'])}<br>'
    dest = f'<b>END:</b> {int(dtf_final.iloc[i]['target'])}<br>'
    dist = f'<b>DIST:</b> {round(dtf_final.iloc[i]['distance'], 2)} m<br>'
    # popup = f'{num}{node}{orig}{dest}{dist}'
    popup = f'{num}{orig}{dest}{dist}'
    if i == 0 :
        folium.Marker(location = [row['y'], row['x']],
                icon = folium.Icon(color = 'red', icon='home'),
                tooltip = popup).add_to(map_start_end)
    elif i == size - 1 :
        folium.Marker(location = [row['y'], row['x']],
                icon = folium.Icon(color = 'green', icon='cloud'),
                tooltip = popup).add_to(map_start_end)

    lines = (row['y'], row['x'])
    route.append(lines)

# Mostrar el mapa
line = folium.PolyLine([route], color = 'cyan', weight = 3.5, opacity = 0.8)
map_start_end.add_child(line)

map_start_end

## Heat map matrix

In [ ]:
heatmap_own = df_distances.copy()
for col in heatmap_own.columns:
    heatmap_own[col] = heatmap_own[col].apply(lambda x: 
        0.3 if pd.isnull(x) else
        (0.7 if np.isinf(x) else 
        (0 if x != 0 else 1)) )
    
fig, ax = plt.subplots(figsize = (10,5))
sns.heatmap(heatmap_own, vmin = 0 , vmax = 1 , cbar = False, ax = ax)
plt.show()

## Maps with graphs and nodes

In [ ]:
lst_routes = dtf_route['route'].tolist()
route_colors = hexa_color_generator_list(len(lst_routes))
print (route_colors)
print(len(lst_routes))
print(lst_routes)
route_linewidths = [4 for _ in  range(len(lst_routes))]
print(route_linewidths)


In [ ]:
print(f'¿G es None?: {G is None}')
if lst_routes:
    print('Primeras rutas:')
    for i in range(0, len(lst_routes)):
        print(f'Ruta {i+1}: {lst_routes[i]}')
        if lst_routes[i] is not None:
            for node in lst_routes[i][:5]: # Verificar los primeros 5 nodos de cada ruta
                print(f'  ¿El nodo {node} existe en G?: {node in G.nodes}')
        else:
            print('  ¡Esta ruta es None!')
else:
    print('La lista de rutas está vacía.')

In [ ]:
fig, ax = ox.plot_graph_routes(G, lst_routes, route_colors = route_colors, route_linewidths = route_linewidths, bgcolor = 'white', node_size = 5)

In [ ]:
df_filtrado = dtf_route.copy()
# df_filtrado = df_filtrado[df_filtrado.apply(lambda x: len(x['route']) > 1, axis = 1)]

In [ ]:
lst_routes = df_filtrado['route'].tolist()
route_colors = hexa_color_generator_list(len(lst_routes))
print (route_colors)
print(len(lst_routes))
print(lst_routes)
route_linewidths = [2 for _ in  range(len(lst_routes))]
# fig, ax = ox.plot_graph_routes(G, lst_routes, bgcolor = 'white', node_size = 1)


In [ ]:
df = pd.DataFrame({
                'origin': 0,
                'target': 0,
                'x': 0,
                'y': 0,
                'x_next': 0,
                'y_next': 0,
                'distance': 0,
                'time_seg': 0},
                index = [0])

tmp = df_animation_multiple_path(G, lst_routes, df)
df = pd.concat([df, tmp], axis = 0)
df = df.reset_index(drop = True)
df = df.drop(index = 0)


In [ ]:
first_node, last_node = lst_routes[0][0], lst_routes[-1][-1]


In [ ]:
df_start = df_filtrado[df_filtrado['origin_node'] == first_node]
df_end = df_filtrado[df_filtrado['destination_node'] == last_node]

In [ ]:
fig = px.scatter_map(data_frame = df, lon = 'x', lat = 'y', zoom = 15, width = 1500, height = 1000, animation_frame = 'id', map_style = 'carto-positron')

fig.data[0].marker = {'size': 12, 'color': 'blue'}

fig.add_trace(px.scatter_map(data_frame = df, lon = 'x', lat = 'y').data[0])
fig.data[1].marker = {'size': 10, 'color': 'black'}

fig.add_trace(px.scatter_map(data_frame = df_start, lon = 'x', lat = 'y').data[0])
fig.data[2].marker = {'size': 15, 'color': 'red'}

fig.add_trace(px.scatter_map(data_frame = df_end, lon = 'x', lat = 'y').data[0])
fig.data[3].marker = {'size': 15, 'color': 'green'}

fig.add_trace(px.line_map(data_frame = df, lon = 'x', lat = 'y').data[0])

fig

# EXTERNAL API's

## Google Maps

In [ ]:
import googlemaps

gmaps = googlemaps.Client(key = 'XXXXXXXXXXXXXXX')

geocode_result = gmaps.geocode('Los Pinos, La Paz, Bolivia')
location = geocode_result[0]['geometry']['location']
print(location.get('lat'))

# Busca restaurantes cercanos
places_result = gmaps.places_nearby(location = location, radius = 500, type = 'restaurant')

print(places_result)
for place in places_result['results']:
    print(place['name'])
    print(place['vicinity'])

# Look up an address with reverse geocoding
reverse_geocode_result = gmaps.reverse_geocode((start[0], start[1]))

# Request directions via public transit
now = datetime.now()
print()
print(f'NOW: {now}')
print(reverse_geocode_result[0])
directions_result = gmaps.directions('Calle 21 de Calacoto',
                                     'Calle 23 de Calacoto',
                                     mode = 'transit',
                                     departure_time = now)
print()
print(directions_result)

# Validate an address with address validation
addressvalidation_result =  gmaps.addressvalidation(['Calle 21 de Calacoto'],
                                                    # regionCode = 'BO',
                                                    locality = 'La Paz', 
                                                    enableUspsCass = True)
print()
print('VALIDATION')
print(addressvalidation_result)



In [ ]:
def get_dist_dur(api_key, start, end):
    base_url = 'https://maps.googleapis.com/maps/api/distancematrix/json'
    params = {
        'origins': start,
        'destinations': end,
        'key': api_key
    }

    response = requests.get(base_url, params = params)

    if response.status_code == 200:
        data = response.json()
        if data['status'] == 'OK':
            print('DATA')
            print(data)
            distance = data['rows'][0]['elements'][0]['distance']['text']
            duration = data['rows'][0]['elements'][0]['duration']['text']
            return distance, duration
        else:
            print('Request failed.')
            return None, None
    else:
        print('Failed to make the request.')
        return None, None


api_key = 'XXXXXXXXXXXXXXXX'
start = 'Plaza Abaroa, Sopocachi, La Paz, Bolivia'
end = 'Cota Cota, La Paz, Bolivia'

distance, duration = get_dist_dur(api_key, start, end)

if distance and duration:
    print(f'Driving Distance: {distance}')
    print(f'Driving Duration: {duration}')

## Map Quest API

In [ ]:
def route_mapquest(api_key, start, end, option, long_x = 0, lat_y = 0):
    api_url = 'http://www.mapquestapi.com'
    route_endpoint = '/directions/v2/route?'
    optimized_endpoint = '/directions/v2/optimizedroute?'
    reverse_endpoint = '/geocoding/v1/reverse?'

    url_route = f'{api_url}{route_endpoint}{urllib.parse.urlencode({'key': api_key, 'from': start, 'to': end})}'    
    url_optimized = f'{api_url}{optimized_endpoint}{urllib.parse.urlencode({'key': api_key,
                    'json': '{"locations":["' + start + '","' + end + '"]}'})}'
    url_reverse = f'{api_url}{reverse_endpoint}{urllib.parse.urlencode({'key': api_key,
                    'location': str(lat_y) + ',' + str(long_x), 'includeRoadMetadata': 'true',
                    'includeNearestIntersection': 'true'})})'
    
    url = url_route if option == 'route' else url_optimized if option == 'optimized' else url_reverse
    print(url)
    
    response = requests.get(url)
    json_data = response.json()
    
    if response.status_code == 200 and option in ('optimization', 'route'):
        trip_duration = json_data['route']['formattedTime']
        distance = json_data['route']['distance'] * 1.61
        # fuel_used = json_data['route']['fuelUsed'] * 3.79
        print('=================================================')
        print(f'Travel info: from: {start} to: {end}.')
        print(f'Trip duration: {trip_duration}.')
        print(f'Distance: {distance:.2f} Km')
        # print('Combustible usado: ' + str('{:.2f}'.format(fuel_used) + ' L'))
        print('=================================================')
        print('Trip Directions:')

        for each in json_data['route']['legs'][0]['maneuvers']:
            distance_remaining = distance - each['distance'] * 1.61 if distance > 0 else 0
            print(each['narrative'])
            print(f'Distance Remaining in Km: {distance_remaining if distance_remaining > 0 else 0:.2f} Km')
            distance = distance_remaining



In [ ]:
api_key = 'XXXXXXXXXXXXX'
start = 'Calle Adrián Patiño, San Miguel, La Paz, Bolivia'
end = 'Calle Andrés Bello, Cota Cota, La Paz, Bolivia'
# -16.541404	-68.076386	-16.541361	-68.076382 -16.539562	-68.069563
route_mapquest(api_key, start, end, 'route', lat_y = -16.539562, long_x = -68.069563)